In [1]:
# ESP32 Decision Tree Training Notebook

# =========================
# Cell 1: Import libraries
# =========================
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import LabelEncoder



In [2]:
# =========================
# Cell 2: Load dataset
# =========================
# Replace this path with your CSV file location
data_path = "/content/health_dataset.csv"
df = pd.read_csv(data_path)
df




,Age,Gender,Weight,Glucose,HeartRate,BMI_Category,Disease
0,24,1,72,58,55,Underweight,Hypoglycemia
1,30,0,65,72,82,Normal,Normal
2,45,1,80,190,95,Overweight,Hyperglycemia
3,55,0,70,165,110,Normal,Tachycardia
4,38,1,90,85,48,Obese,Bradycardia
...,...,...,...,...,...,...,...
1170,55,0,104,139,52,Obese,Bradycardia
1171,48,1,94,75,110,Normal,Tachycardia
1172,43,0,78,152,95,Normal,Hyperglycemia
1173,60,1,91,62,69,Overweight,Hypoglycemia


In [3]:
# =========================
# Cell 3: Sanity check
# =========================
print(df.info())
print(df.isnull().sum())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1175 entries, 0 to 1174
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Age           1175 non-null   object
 1   Gender        1175 non-null   object
 2   Weight        1175 non-null   object
 3   Glucose       1175 non-null   object
 4   HeartRate     1175 non-null   object
 5   BMI_Category  1175 non-null   object
 6   Disease       1175 non-null   object
dtypes: object(7)
memory usage: 64.4+ KB
None
Age             0
Gender          0
Weight          0
Glucose         0
HeartRate       0
BMI_Category    0
Disease         0
dtype: int64


In [4]:
# =========================
# Cell 4: Encode categorical features
# =========================
label_encoders = {}
for col in ["BMI_Category", "Disease"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

df



,Age,Gender,Weight,Glucose,HeartRate,BMI_Category,Disease
0,24,1,72,58,55,4,3
1,30,0,65,72,82,1,4
2,45,1,80,190,95,3,2
3,55,0,70,165,110,1,6
4,38,1,90,85,48,2,0
...,...,...,...,...,...,...,...
1170,55,0,104,139,52,2,0
1171,48,1,94,75,110,1,6
1172,43,0,78,152,95,1,2
1173,60,1,91,62,69,3,3


In [5]:
# =========================
# Cell 5: Define features and target
# =========================
X = df.drop("Disease", axis=1)
y = df["Disease"]
X, y



(     Age Gender Weight Glucose HeartRate  BMI_Category
 0     24      1     72      58        55             4
 1     30      0     65      72        82             1
 2     45      1     80     190        95             3
 3     55      0     70     165       110             1
 4     38      1     90      85        48             2
 ...   ..    ...    ...     ...       ...           ...
 1170  55      0    104     139        52             2
 1171  48      1     94      75       110             1
 1172  43      0     78     152        95             1
 1173  60      1     91      62        69             3
 1174  38      0     75     114       103             1
 
 [1175 rows x 6 columns],
 0       3
 1       4
 2       2
 3       6
 4       0
        ..
 1170    0
 1171    6
 1172    2
 1173    3
 1174    6
 Name: Disease, Length: 1175, dtype: int64)

In [7]:
# =========================
# Cell 6: Train Decision Tree
# =========================
# Convert relevant columns in X to numeric type.
# The previous cells loaded these columns as 'object' (string) type,
# and the DecisionTreeClassifier expects numerical input.
numeric_cols_to_convert = ['Age', 'Gender', 'Weight', 'Glucose', 'HeartRate']
for col in numeric_cols_to_convert:
    # Attempt to convert to numeric, coercing errors to NaN if any non-numeric value exists
    X[col] = pd.to_numeric(X[col], errors='coerce')

# Fill any NaN values that resulted from coercion (if any non-numeric data was present)
# For simplicity, filling with the mean of the respective column.
X = X.fillna(X.mean(numeric_only=True))

model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, random_state=42)
model.fit(X, y)

DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, random_state=42)

In [8]:
# =========================
# Cell 7: Inspect tree rules
# =========================
tree_rules = export_text(model, feature_names=list(X.columns))
print(tree_rules)

|--- Glucose <= 141.50
|   |--- HeartRate <= 100.50
|   |   |--- HeartRate <= 70.50
|   |   |   |--- Glucose <= 77.50
|   |   |   |   |--- class: 3
|   |   |   |--- Glucose >  77.50
|   |   |   |   |--- class: 0
|   |   |--- HeartRate >  70.50
|   |   |   |--- Glucose <= 71.50
|   |   |   |   |--- class: 3
|   |   |   |--- Glucose >  71.50
|   |   |   |   |--- class: 4
|   |--- HeartRate >  100.50
|   |   |--- Glucose <= 128.50
|   |   |   |--- Glucose <= 64.00
|   |   |   |   |--- class: 3
|   |   |   |--- Glucose >  64.00
|   |   |   |   |--- class: 6
|   |   |--- Glucose >  128.50
|   |   |   |--- HeartRate <= 105.00
|   |   |   |   |--- class: 6
|   |   |   |--- HeartRate >  105.00
|   |   |   |   |--- class: 2
|--- Glucose >  141.50
|   |--- HeartRate <= 55.00
|   |   |--- class: 0
|   |--- HeartRate >  55.00
|   |   |--- Glucose <= 142.50
|   |   |   |--- Age <= 51.50
|   |   |   |   |--- class: 2
|   |   |   |--- Age >  51.50
|   |   |   |   |--- class: 4
|   |   |--- Glucose > 

In [9]:
# =========================
# Cell 8: Map class labels back
# =========================
disease_mapping = dict(zip(label_encoders["Disease"].classes_,
                           label_encoders["Disease"].transform(label_encoders["Disease"].classes_)))
disease_mapping



{'Bradycardia': np.int64(0),
 'Disease': np.int64(1),
 'Hyperglycemia': np.int64(2),
 'Hypoglycemia': np.int64(3),
 'Normal': np.int64(4),
 'Tachy': np.int64(5),
 'Tachycardia': np.int64(6)}

In [12]:
# =========================
# Cell 9: Test prediction
# =========================
sample = np.array([[35, 1, 78, 100, 150, 1]])  # example input
pred = model.predict(sample)
pred, label_encoders["Disease"].inverse_transform(pred)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


(array([6]), array(['Tachycardia'], dtype=object))

In [16]:
pip install micromlgen


  Preparing metadata (setup.py) ... done
  Created wheel for micromlgen: filename=micromlgen-1.1.28-py3-none-any.whl size=32152 sha256=029d59dc445ecff31a2e20f196b06492aa2e666810102144acec23d39d49e881
  Stored in directory: /root/.cache/pip/wheels/16/02/8a/3a8b533174e4f7691a8fd72dab4493fb6819b79f8fcc1d18a6
Successfully built micromlgen


In [19]:
from micromlgen import port

# model: your trained DecisionTreeClassifier
# convert labels to a dictionary mapping int -> string
classmap = {i: label for i, label in enumerate(label_encoders["Disease"].classes_)}

# generate C code
c_code = port(model, classmap=classmap)

# save to header file
with open("decision_tree_model.h", "w") as f:
    f.write(c_code)

print("C header file created successfully!")


C header file created successfully!
